# Head konteks epoch 5 → Core ML FP32

Buka dari root **SongMap**, lalu **Run All** di Mac. Notebook memanggil Python sibling **Section-Annotations/.venv-songformer311** secara eksplisit.

Kita mengonversi **head tambahan** yang sudah dikunci, lalu memeriksa hasil Core ML pada 32 lagu validation dengan input palette6 yang sama. Acuannya adalah akurasi **79,16%** dan macro F1 **0,7672** pada validation tersebut, bukan angka 80% dari test.

Paket menerima mean fitur audio `[N,4096]` dan timing `[N,6]` untuk **seluruh section asli**, lalu mengeluarkan koreksi logits `[N,8]`. Probabilitas head SongFormer lama tetap digabungkan di luar paket dengan rumus `log(max(p,1e-8)) + residual`, disusul softmax delapan kelas. Encoder, head SongFormer palette6, label dan boundary tidak berubah.

FP32 dan CPU_ONLY dipakai sebagai acuan paritas. Paket baru belum dipasang ke aplikasi dan belum diukur di iPhone.


In [ ]:
from pathlib import Path
from datetime import datetime
import subprocess, json

SONGMAP_ROOT=Path.cwd().resolve()
assert (SONGMAP_ROOT/'scripts/context_coreml.py').is_file(), 'Buka notebook dari root SongMap'
SONGSTRUCT_ROOT=(SONGMAP_ROOT/'../Section-Annotations').resolve()
RUN_PYTHON=SONGSTRUCT_ROOT/'.venv-songformer311/bin/python'
assert RUN_PYTHON.is_file(), f'Python tidak ditemukan: {RUN_PYTHON}'
files=list((SONGMAP_ROOT/'outputs.local').glob('context-palette6-validation-*/validation-all.json'))
assert files, 'Selesaikan section_context_palette6_validation.ipynb dahulu'
PALETTE_VALIDATION=max(files,key=lambda p:p.stat().st_mtime).resolve()
previous=json.loads(PALETTE_VALIDATION.read_text())
assert previous['full_validation'] and previous['selected_epoch']==5
selection=json.loads((PALETTE_VALIDATION.parent/'selection.json').read_text())
frozen_files=list((SONGMAP_ROOT/'outputs.local').glob('frozen-context-test-*/selection.json'))
FROZEN=next((p.parent.resolve() for p in frozen_files
             if json.loads(p.read_text())['identity']==selection['frozen_selection_identity']),None)
assert FROZEN is not None, 'Snapshot asli dari evaluasi epoch 5 tidak ditemukan'
cache=SONGMAP_ROOT/'outputs.local/section-context-palette6-cache-v1'
output=SONGMAP_ROOT/'outputs.local'/('context-coreml-fp32-'+datetime.now().strftime('%Y%m%d-%H%M%S'))
export_dir=output/'export'
parity_dir=output/'parity'
subprocess.run([str(RUN_PYTHON),'-c',
    "import sys,numpy,torch,coremltools; assert sys.platform=='darwin','Jalankan native Core ML di Mac'; print('Python:',sys.executable); print('Torch:',torch.__version__,'Core ML Tools:',coremltools.__version__)"],check=True)
print('Validation acuan:',PALETTE_VALIDATION)
print('Snapshot terkunci:',FROZEN)
print('F1 acuan:',previous['palette6_features_with_context']['macro_f1_present'])
print('Output:',output)


In [ ]:
def run_logged(command,log):
    log.parent.mkdir(parents=True,exist_ok=True)
    with log.open('a',encoding='utf-8') as handle:
        process=subprocess.Popen(command,cwd=SONGMAP_ROOT,stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,text=True,bufsize=1)
        try:
            for line in process.stdout:
                print(line,end='');handle.write(line);handle.flush()
            status=process.wait()
        except KeyboardInterrupt:
            process.terminate();process.wait();raise
    if status:
        raise RuntimeError(f'Gagal ({status}); lihat {log}. Diagnostik tersimpan; jangan memasang paket sebelum paritas lulus.')

def base_command(action,destination):
    return [str(RUN_PYTHON),'-u',str(SONGMAP_ROOT/'scripts/context_coreml.py'),action,
        '--songstruct-root',str(SONGSTRUCT_ROOT),'--palette-validation',str(PALETTE_VALIDATION),
        '--frozen-evaluation',str(FROZEN),'--cache-dir',str(cache),'--output-dir',str(destination)]

if (export_dir/'export_report.json').is_file():
    saved=json.loads((export_dir/'export_report.json').read_text())
    assert saved['native_synthetic_parity_passed'], 'Ekspor sebelumnya gagal paritas; periksa diagnostiknya'
    print('Menggunakan ekspor yang sudah tersedia:',saved['package'])
else:
    run_logged(base_command('export',export_dir),Path(str(output)+'.export.log'))
export_report=json.loads((export_dir/'export_report.json').read_text())
print('Paket:',export_report['package'])
print('Ukuran:',f"{export_report['package_bytes']/1024**2:.2f} MiB")
print('Deklarasi tensor:',export_report['declared_tensor_dtypes'])


Ekspor memeriksa bentuk attention operasi dasar terhadap model PyTorch asli pada input nyata, trace dengan beberapa panjang input, serta inferensi Core ML pada input sintetis yang jumlah section-nya berubah. Batas paket adalah 1–512 section; section tidak dipotong agar muat.

Cell berikut menjalankan satu lagu lalu seluruh validation dengan paket yang sama. Periksaannya mencakup **residual, logits akhir, probabilitas, dan label**, sehingga label yang sama saja belum cukup untuk dinyatakan lulus. Toleransi FP32 ditetapkan sebelum pengukuran; kegagalan disimpan sebagai diagnostik, bukan persetujuan integrasi.


In [ ]:
def verify(limit):
    return base_command('verify',parity_dir)+['--export-dir',str(export_dir),'--limit',str(limit)]

run_logged(verify(1),Path(str(output)+'.verify-one.log'))


In [ ]:
if (parity_dir/'validation-all.json').is_file():
    saved=json.loads((parity_dir/'validation-all.json').read_text())
    assert saved['passed'], 'Paritas sebelumnya gagal; periksa checks dalam laporan'
    print('Paritas seluruh validation sudah tersedia')
else:
    run_logged(verify(0),Path(str(output)+'.verify-all.log'))


In [ ]:
report=json.loads((parity_dir/'validation-all.json').read_text())
print('Status:',report['status'],'| lagu:',report['tracks'],'| seluruh validation:',report['full_validation'])
for name in ('pytorch','coreml'):
    score=report[name]
    print(f"{name}: accuracy={score['section_accuracy']:.2%}; macro F1={score['macro_f1_present']:.6f}")
print('Kesamaan label:',f"{report['section_agreement']:.4%}")
print('Delta F1:',report['macro_f1_delta'])
for kind in ('residual','final_logits','probabilities'):
    worst=max(check[kind].get('max_abs_error',float('inf')) for check in report['checks'])
    print('Max error',kind,':',worst)
print('Paket:',export_report['package'])
print('Kirim laporan:',parity_dir/'validation-all.json')
print('Paket masih lokal di outputs.local; belum dipasang ke Resources/MLModels aplikasi.')


Jika proses verify terputus, jalankan ulang cell verify dengan variabel output yang sama. Hasil per lagu disimpan dengan identitas paket, input, runtime dan toleransi. Untuk mempertahankan folder setelah kernel ditutup, isi ulang variabel `output`, `export_dir`, dan `parity_dir` setelah setup.

Bagikan **parity/validation-all.json**. Kelulusan ini hanya membuktikan paritas head konteks Core ML CPU pada input validation tersimpan. Uji integrasi pooling audio dan pengukuran iPhone tetap diperlukan.
